# Fine-tune Qwen 3.5 0.8B for Text-to-SQL

We teach a small model (**Qwen 3.5, 0.8B parameters**) to turn a question about a table into a SQL query.
Everything runs locally on a Mac with **MLX**: load the data, test the base model, fine-tune it with **LoRA**, and measure the difference.

In [1]:
import json
import random
from pathlib import Path

from huggingface_hub import hf_hub_download
from mlx_lm import load, generate

## 0. Check the machine

Everything in this notebook runs on a **MacBook Pro**. MLX trains on the Mac's own GPU through Metal: no NVIDIA GPU, no CUDA, no cloud.

In [2]:
!system_profiler SPHardwareDataType | grep -E "Model Name|Chip|Memory"

      Model Name: MacBook Pro
      Chip: Apple M5 Max
      Memory: 36 GB


In [3]:
import mlx.core as mx
mx.default_device()

Device(gpu, 0)

## 1. Load the dataset

[`b-mc2/sql-create-context`](https://huggingface.co/datasets/b-mc2/sql-create-context) has about 78k examples. Each one has three fields:

- `question`: a question in plain English
- `context`: the `CREATE TABLE` statement of the table it asks about
- `answer`: the correct SQL query

In [4]:
path = hf_hub_download("b-mc2/sql-create-context", "sql_create_context_v4.json",
                       repo_type="dataset")
path

'/Users/laxmi/.cache/huggingface/hub/datasets--b-mc2--sql-create-context/snapshots/9d80a6a118b838d9defc3798d659a54a2ac2ff37/sql_create_context_v4.json'

In [5]:
rows = json.load(open(path))
len(rows)

78577

In [6]:
rows[0]

{'question': 'How many heads of the departments are older than 56 ?',
 'context': 'CREATE TABLE head (age INTEGER)',
 'answer': 'SELECT COUNT(*) FROM head WHERE age > 56'}

## 2. Split into train, validation and test

We shuffle once with a fixed seed, so the split is the same on every run. The first 100 examples are the **test set**: the model never sees them during training, so they tell us how well it really learned.

In [7]:
random.seed(42)
random.shuffle(rows)
test, valid, train = rows[:100], rows[100:300], rows[300:2300]
len(train), len(valid), len(test)

(2000, 200, 100)

## 3. Turn each example into a chat

Qwen is a chat model, so every example becomes a short conversation. The **user** message holds the table and the question, the **assistant** message holds the SQL. `mlx_lm.lora` trains on files of these, one JSON object per line.

In [8]:
def prompt(ex):
    return (f"{ex['context']}\n\nQuestion: {ex['question']}\n"
            "Write a SQLite query that answers the question. Return only the SQL query.")

In [9]:
def to_chat(ex):
    return {"messages": [{"role": "user", "content": prompt(ex)},
                         {"role": "assistant", "content": ex["answer"]}]}

to_chat(train[0])

{'messages': [{'role': 'user',
   'content': "CREATE TABLE table_21458142_1 (sales_breakdown VARCHAR, publisher VARCHAR, title VARCHAR)\n\nQuestion: What's the sales breakdown for Nintendo's Mario Kart DS?\nWrite a SQLite query that answers the question. Return only the SQL query."},
  {'role': 'assistant',
   'content': 'SELECT sales_breakdown FROM table_21458142_1 WHERE publisher = "Nintendo" AND title = "Mario Kart DS"'}]}

In [10]:
Path("data").mkdir(exist_ok=True)
for name, part in [("train", train), ("valid", valid)]:
    with open(f"data/{name}.jsonl", "w") as f:
        f.writelines(json.dumps(to_chat(ex)) + "\n" for ex in part)

## 4. Try the base model

Before training, let's see what the original model does with a test question. We turn thinking off: we want the query, not the reasoning.

In [11]:
model, tokenizer = load("mlx-community/Qwen3.5-0.8B-bf16")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

In [12]:
def ask(model, ex):
    messages = [{"role": "user", "content": prompt(ex)}]
    text = tokenizer.apply_chat_template(messages, add_generation_prompt=True,
                                         tokenize=False, enable_thinking=False)
    return generate(model, tokenizer, prompt=text, max_tokens=200)

In [13]:
ex = test[80]
print(ex["question"])
print(ask(model, ex))

What is the abbreviation of Airline "JetBlue Airways"?
SELECT DISTINCT Airline FROM AIRLINES WHERE Airline = 'JetBlue Airways';


In [14]:
ex["answer"]

'SELECT Abbreviation FROM AIRLINES WHERE Airline = "JetBlue Airways"'

## 5. Measure it

Comparing SQL text is too strict: two different queries can both be right. So we **run** each query. `sql_eval.py` fills small SQLite databases with random rows, runs the model's query and the correct one, and checks that they return the same rows.

In [15]:
from sql_eval import check, extract_sql
from tqdm.auto import tqdm

def evaluate(model):
    results = [check(ex, extract_sql(ask(model, ex))) for ex in tqdm(test)]
    return results.count("correct")

In [16]:
evaluate(model)

  0%|          | 0/100 [00:00<?, ?it/s]

44

## 6. Fine-tune with LoRA

LoRA keeps the model's own weights **frozen** and trains small extra matrices next to some of its layers, a tiny fraction of the parameters. That is why it fits on a laptop, and the result is a small **adapter** file.

`--mask-prompt` computes the loss only on the assistant's answer, so the model learns to write SQL, not to repeat the question. `--grad-checkpoint` recomputes some values during the backward pass instead of keeping them in memory: training needs far less memory, at almost the same speed.

`/usr/bin/time -l` is the Mac's built-in timer: it saves how long training took and its peak memory to `train_usage.txt`.

In [17]:
!/usr/bin/time -l -o train_usage.txt \
    mlx_lm.lora --model mlx-community/Qwen3.5-0.8B-bf16 --train --data data \
    --iters 500 --batch-size 4 --mask-prompt --grad-checkpoint --adapter-path adapters

Loading pretrained model
Reconstructing (incomplete total...): |           |  0.00B /  0.00B            

Fetching 10 files: 100%|█████████████████████| 10/10 [00:00<00:00, 2718.10it/s]
Download complete:                                         |  0.00B            
Download complete:                                         |  0.00B            
Reconstruction complete: |                        |  0.00B /  0.00B            
Loading datasets
Training
╭─ mlx_lm.lora ────────────────────────────────────────────────────────────────╮
│    model           mlx-community/Qwen3.5-0.8B-bf16                           │
│    type            lora · 16 layers · rank 8                                 │
│    dataset         data                                                      │
│    optimizer       adam · lr 1.0e-05                                         │
│    batch · iters   4 · 500                                                   │
│    max seq         2,048                                    

## 7. Try the tuned model

Same model, same question, now with the adapter loaded on top.

In [18]:
tuned, _ = load("mlx-community/Qwen3.5-0.8B-bf16", adapter_path="adapters")
print(ask(tuned, ex))

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

SELECT Abbreviation FROM AIRLINES WHERE Airline = "JetBlue Airways"


## 8. Before vs after

`compare.py` runs the base model and the fine-tuned model on the same 100 test questions, and reads the training time and memory from `train_usage.txt`.

In [19]:
!python compare.py

Reconstructing (incomplete total...): |           |  0.00B /  0.00B            

Fetching 10 files: 100%|█████████████████████| 10/10 [00:00<00:00, 2984.42it/s]
Download complete:                                         |  0.00B            
Download complete:                                         |  0.00B            
Reconstruction complete: |                        |  0.00B /  0.00B            
Reconstructing (incomplete total...): |           |  0.00B /  0.00B            

Fetching 10 files: 100%|█████████████████████| 10/10 [00:00<00:00, 3432.89it/s]
Download complete:                                         |  0.00B            
Download complete:                                         |  0.00B            
Reconstruction complete: |                        |  0.00B /  0.00B            
Machine       MacBook Pro · Apple M5 Max · 36 GB unified memory
GPU           Apple M5 Max through Metal (MLX) · no NVIDIA GPU, no CUDA

Fine-tuning   LoRA · 500 steps · 2,000 examples
Train time   

## 9. Fuse the adapter

Fusing merges the adapter into the weights. The result is one normal model folder that you can load, share or serve without the adapter.

In [20]:
!mlx_lm.fuse --model mlx-community/Qwen3.5-0.8B-bf16 --adapter-path adapters \
    --save-path qwen3.5-0.8b-sql

Loading pretrained model
Reconstructing (incomplete total...): |           |  0.00B /  0.00B            

Fetching 10 files: 100%|█████████████████████| 10/10 [00:00<00:00, 3391.25it/s]
Download complete:                                         |  0.00B            
Download complete:                                         |  0.00B            
Reconstruction complete: |                        |  0.00B /  0.00B            


## That's it

A 0.8B model, fine-tuned on a MacBook Pro in about 10 minutes, without an NVIDIA GPU.

If this helped you, **like the video** and **subscribe** for more hands-on AI tutorials.